In [0]:
dbutils.widgets.text("catalogo", "electrocasa_dev", "Catálogo a validar")
catalogo = dbutils.widgets.get("catalogo")
spark.sql(f"USE CATALOG {catalogo}")

tablas_obligatorias = [
    "bronze.ventas", "bronze.catalogo", "bronze.empleados", "bronze.resenas",
    "bronze.devoluciones", "bronze.tracking_envios", "bronze.sucursales",
    "silver.ventas", "silver.catalogo", "silver.empleados", "silver.resenas",
    "silver.devoluciones", "silver.tracking_envios", "silver.sucursales",
    "silver.cuarentena",
    "gold.ventas_netas_sucursal_mes", "gold.top_productos", "gold.dotacion_mensual",
    "gold.ventas_vendedor", "gold.resenas_negativas", "gold.despachos_courier"
]

print(f"=== VALIDACIÓN DE CALIDAD EN {catalogo} ===")
for t in tablas_obligatorias:
    conteo = spark.table(t).count()
    print(f"OK | {t}: {conteo} filas")
    assert conteo > 0, f"ERROR CRÍTICO: La tabla {t} está vacía."

# Verificar historia SCD Type 2 en empleados (deben existir intervalos cerrados con __END_AT)
cerrados = spark.table("silver.empleados").filter("__END_AT IS NOT NULL").count()
print(f"OK | Versiones históricas cerradas SCD2 en silver.empleados: {cerrados}")
assert cerrados > 0, "ERROR: No se generaron intervalos históricos (__END_AT) en silver.empleados."

# Resumen de registros enviados a cuarentena por fuente
display(spark.sql("""
    SELECT fuente, regla_fallida, COUNT(*) AS filas_rechazadas
    FROM silver.cuarentena
    GROUP BY fuente, regla_fallida
    ORDER BY filas_rechazadas DESC
"""))